# Solución Ejercicio 03c  |  Tema 4 - PRyES
## Intervalos de confianza
stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

### Se dispone de una muestra de tamaño 20 de la variable aleatoria X, y se quiere calcular un intervalo de confianza al 90% en tres supuestos diferentes:

1. X se distribuye como una normal de varianza 1.35
2. X se distribuye como una normal, pero de varianza desconocida
3. No se sabe qué distribución puede seguir la variable X

### La muestra almacenada en el fichero muestra_n20v02y2022.csv se guarda en la variable x con el siguiente código para facilitar trabajar con ella: 

In [ ]:
# cargamos los datos de la muestra disponible

df_orig = pd.read_csv('muestra_n20v02y2022.csv')
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)

# datos de la muestra para trabajar
x = df_orig.x.tolist()    #   <-------------------------------- x (valores en la muestra)
n = len(x)                #   <-------------------------------- n (tamaño de la muestra)

# observamos su distribución
plt.figure
plt.hist(x);plt.xlabel('x');plt.title(f'Muestra (n={n})')
plt.show()

### a) Calcula un intervalo de confianza para la media poblacional de la v.a. X con un nivel de confianza del 90% en el primer supuesto. Explica claramente los pasos seguidos para crearlo.

In [ ]:
# Respuesta

print('En el primer supuesto asumimos que X sigue una Normal de varianza 1.35')
print('En esta situación el mejor estimador de intervalo se apoya en el estadístico de contraste '
      'media muestral, que se distribuye como una normal de desviación típica sigma_pobl/n')

# media muestral
x_media = np.mean(x)

#  error estándar del estimador
var_pob = 1.35
var_estim = var_pob / n
se_estim = np.sqrt(var_estim)

# Ahora creamos la v.a. MEDIA MUESTRAL como una normal
Xmedia_supuesto1 = stats.norm(loc=x_media, scale = se_estim)  

#Para calcular un IC de la media de X usamos la distribución teórica de la media muestral
gamma = 0.90
alfa = 1 - gamma
lim_inf_supuesto1 = Xmedia_supuesto1.ppf(alfa/2)
lim_sup_supuesto1 = Xmedia_supuesto1.ppf(1-alfa/2)

print(f'El intervalo al {100*gamma}% de la media de X en el primer supuesto es: [{lim_inf_supuesto1}, {lim_sup_supuesto1}]')

### b) Calcula un intervalo de confianza para la media poblacional de la v.a. X con un nivel de confianza del 90% en el segundo supuesto. Explica claramente los pasos seguidos para crearlo.

In [ ]:
# Respuesta

print('En el segundo supuesto se considera que X sigue una Normal, pero de varianza desconocida.')
print('Además, la muestra es de un tamaño pequeño, por lo que es importante emplear el estimador '
      'de intervalo apropiado. En este caso es necesario estimar a partir de la muestra la varianza '
      'poblacional y utilizarla en el cálculo del IC.')

# media muestral
x_media = np.mean(x)

# calculamos la desviación típica muestral sx sin sesgo (dividiendo por n-1)
sx = np.std(x, ddof=1)

# Ahora creamos la v.a. MEDIA MUESTRAL como una t de student
Xmedia_supuesto2 = stats.t(n-1, loc=x_media, scale = sx/np.sqrt(n))  

#Para calcular un IC de la media de X usamos la distribución teórica de la media muestral
gamma = 0.90
alfa = 1 - gamma
lim_inf_supuesto2 = Xmedia_supuesto2.ppf(alfa/2)
lim_sup_supuesto2 = Xmedia_supuesto2.ppf(1-alfa/2)

print(f'El intervalo al {100*gamma}% de la media de X en el segundo supuesto es: [{lim_inf_supuesto2}, {lim_sup_supuesto2}]')

### c) Calcula un intervalo de confianza para la media poblacional de la v.a. X con un nivel de confianza del 90% en el tercer y último supuesto. Explica claramente los pasos seguidos para crearlo.

In [ ]:
# Respuesta

print('En el tercer supuesto no se sabe cómo se distribuye X.')
print('En esta situación no se puede calcular el IC a partir de una expresión analítica, '
      'pero se puede hacer utilizando bootstrap.')

# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (x,)  # tiene que ser una secuencia
R = 9999
estadistico = np.mean
nivel_confianza = 0.90

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf_supuesto3, lim_sup_supuesto3 = res_BCa.confidence_interval
SE_BCa = res_BCa.standard_error

print(f'IC {100*nivel_confianza}% estimado según método BCa para la media de X en el tercer supuesto: [{lim_inf_supuesto3}, {lim_sup_supuesto3}]')

### d) Representa en la misma gráfica, los tres intervalos de confianza construidos para los supuestos considerados. Calcula el ancho de los tres y relacionalos con las hipótesis de los tres supuestos.

In [ ]:
# Respuesta

# representamos los tres intervalos juntos

plt.figure(1, figsize=(15, 4))

plt.errorbar(x_media, 1, xerr = np.array([[x_media - lim_inf_supuesto1], [lim_sup_supuesto1 - x_media]]), fmt = 'o', label = f'IC [{lim_inf_supuesto1}, {lim_sup_supuesto1}] supuesto 1')
plt.errorbar(x_media, 2, xerr = np.array([[x_media - lim_inf_supuesto2], [lim_sup_supuesto2 - x_media]]), fmt = 'o', label = f'IC [{lim_inf_supuesto2}, {lim_sup_supuesto2}] supuesto 2')
plt.errorbar(x_media, 3, xerr = np.array([[x_media - lim_inf_supuesto3], [lim_sup_supuesto3 - x_media]]), fmt = 'o', label = f'IC [{lim_inf_supuesto3}, {lim_sup_supuesto3}] supuesto 3')

plt.title('IC para la media de X en los tres supuestos')
plt.tight_layout()

plt.ylim(0, 5)

plt.legend()
plt.grid()
plt.show()

# calculamos anchos
ancho_supuesto1 = lim_sup_supuesto1 - lim_inf_supuesto1
ancho_supuesto2 = lim_sup_supuesto2 - lim_inf_supuesto2
ancho_supuesto3 = lim_sup_supuesto3 - lim_inf_supuesto3

print(f'Ancho supuesto 1 (X sigue una Normal con varianza conocida): {ancho_supuesto1}')
print(f'Ancho supuesto 2 (X sigue una Normal con varianza desconocida): {ancho_supuesto2}')
print(f'Ancho supuesto 3 (X con distribución desconocida): {ancho_supuesto3}')

print(f'En el supuesto 2 el ancho es menor que en el primer supuesto porque el segundo se ha estimado '
      'una varianza {sx} a partir de la muestra, menor que la varianza de 1.35 considerada en el primero. '
      'El ancho del IC en el supuesto 3, usando bootstrap, es el menor de todos ellos, quizás estamos infraestimando la variabilidad muestral '
      'al partir de una muestra de tamaño pequeño.')